In [1]:
import pandas as pd
import duckdb
import os
import os, json
from uuid import uuid4
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

In [21]:
conn = duckdb.connect("/srv/data/grela/grela_v0-4.duckdb")
#conn.execute("CREATE TABLE works AS SELECT * FROM works_df")

In [3]:
# make a simple query to extract all works or a subset of works
query = """
SELECT *
FROM works
/*
    alternatively uncomment this:
    WHERE grela_id LIKE 'vulgate_tlg0031%' OR grela_id LIKE 'vulgate_tlg0527%';
*/
"""
works_df = conn.execute(query).fetchdf()

In [16]:
len(works_df[works_df["textsource"]=="exprecce"])

25

In [4]:
conn.execute("""
ALTER TABLE works ADD COLUMN IF NOT EXISTS textsource VARCHAR;
""")

In [5]:
sentences_info = conn.execute(f"""
        PRAGMA table_info(sentences)
    """).fetchall()
sentences_info

[(0, 'sentence_id', 'VARCHAR', False, None, False),
 (1, 'grela_id', 'VARCHAR', False, None, False),
 (2, 'position', 'INTEGER', False, None, False),
 (3, 'text', 'VARCHAR', False, None, False),
 (4, 'subwork_id', 'VARCHAR', False, None, False)]

In [6]:
tokens_info = conn.execute(f"""
        PRAGMA table_info(tokens)
    """).fetchall()
tokens_info

[(0, 'sentence_id', 'VARCHAR', False, None, False),
 (1, 'grela_id', 'VARCHAR', False, None, False),
 (2, 'token_text', 'VARCHAR', False, None, False),
 (3, 'lemma', 'VARCHAR', False, None, False),
 (4, 'pos', 'VARCHAR', False, None, False),
 (5, 'char_start', 'INTEGER', False, None, False),
 (6, 'char_end', 'INTEGER', False, None, False),
 (7, 'token_id', 'BIGINT', False, None, False),
 (8, 'ref', 'JSON', False, None, False)]

In [7]:
result = conn.execute(f"""
        PRAGMA table_info(tokens)
    """).fetchall()

    # Extract column names
columns = [row[1] for row in result]

if "ref" not in columns:
    conn.execute(f"ALTER TABLE tokens ADD COLUMN ref JSON")
    conn.execute(f"UPDATE tokens SET ref = '{{}}'")


In [8]:
import os, json, pickle
from pathlib import Path
import pandas as pd

def process_single_pickle(pickle_path: Path, grela_prefix: str, jsonize_ref: bool = True):
    """Load ONE .pickle file and return sentences_df, tokens_df for that grela_id."""
    base_id = pickle_path.stem
    grela_id = f"{grela_prefix}_{base_id}"

    with open(pickle_path, "rb") as f:
        sents_data = pickle.load(f)

    sentences = []
    tokens = []

    for sent in sents_data:
        # works with either 4-tuple or 5-tuple (if cts_source is appended)
        work_id, pos, text, token_data = sent[:4]
        sentence_id = f"{grela_id}_{pos}"
        sentences.append([sentence_id, grela_id, pos, text, ""])  # subwork_id empty for now

        for tok in token_data:
            tok_text, lemma, pos_tag, ref, char_start, char_end = tok[:6]
            if jsonize_ref and isinstance(ref, dict):
                ref_val = json.dumps(ref, ensure_ascii=False)
            else:
                # DuckDB doesn’t like pandas “object” dicts much; JSON string is faster
                ref_val = ref if isinstance(ref, (str, type(None))) else None

            tokens.append([
                sentence_id,
                grela_id,
                tok_text,
                lemma,
                pos_tag,
                int(char_start),
                int(char_end),
                None,         # token_id (optional, keep None)
                ref_val,      # JSON-ified ref for speed
            ])

    sentences_df = pd.DataFrame(
        sentences,
        columns=["sentence_id", "grela_id", "position", "text", "subwork_id"],
    )
    # Optional: enforce dtypes for speed
    sentences_df = sentences_df.astype({
        "sentence_id": "string",
        "grela_id": "string",
        "position": "int32",
        "text": "string",
        "subwork_id": "string",
    })

    tokens_df = pd.DataFrame(
        tokens,
        columns=["sentence_id", "grela_id", "token_text", "lemma", "pos",
                 "char_start", "char_end", "token_id", "ref"],
    )
    tokens_df = tokens_df.astype({
        "sentence_id": "string",
        "grela_id": "string",
        "token_text": "string",
        "lemma": "string",
        "pos": "string",
        "char_start": "int32",
        "char_end": "int32",
        # token_id stays nullable; ref is JSON string or None
    })

    return grela_id, sentences_df, tokens_df

In [9]:
%%time
SOURCE_MAP = {
    "/srv/data/greek/exprecce_sentences_2025-10/": "exprecce",
    #"/srv/data/greek/glaux_sentences_2025-08/":    "glaux",
    #"/srv/data/greek/oga_sentences_2025-08/":      "oga",
}

data_sources = [
    ("/srv/data/greek/exprecce_sentences_2025-10/", "lagt"),
    #("/srv/data/greek/glaux_sentences_2025-08/",    "lagt"),
    #("/srv/data/greek/oga_sentences_2025-08/",      "lagt"),
]

seen_grela_ids = set()

conn.execute("BEGIN;")
try:
    for dir_path, prefix in data_sources:
        textsource = SOURCE_MAP[dir_path]
        for i, pickle_path in enumerate(sorted(Path(dir_path).glob("*.pickle")), start=1):
            base_id = pickle_path.stem
            grela_id = f"{prefix}_{base_id}"

            if grela_id in seen_grela_ids:
                continue

            grela_id_check, sents_df, toks_df = process_single_pickle(pickle_path, prefix, jsonize_ref=True)
            assert grela_id_check == grela_id

            # refresh rows for this grela_id
            conn.execute("DELETE FROM sentences WHERE grela_id = ?", [grela_id])
            conn.execute("DELETE FROM tokens    WHERE grela_id = ?", [grela_id])

            conn.register("temp_sentences", sents_df)
            conn.execute("INSERT INTO sentences SELECT * FROM temp_sentences")
            conn.unregister("temp_sentences")

            conn.register("temp_tokens", toks_df)
            conn.execute("INSERT INTO tokens SELECT * FROM temp_tokens")
            conn.unregister("temp_tokens")

            # Upsert works.textsource, but do NOT overwrite if it’s already set
            conn.execute(
                "UPDATE works SET textsource = ? "
                "WHERE grela_id = ? AND (textsource IS NULL OR textsource = '')",
                [textsource, grela_id],
            )

            # after inserting sentences/tokens for this grela_id
            conn.execute("UPDATE works SET textsource = ? WHERE grela_id = ?", [textsource, grela_id])
            conn.execute(
                "INSERT INTO works (grela_id, textsource) "
                "SELECT ?, ? WHERE NOT EXISTS (SELECT 1 FROM works WHERE grela_id = ?)",
                [grela_id, textsource, grela_id],
            )

            seen_grela_ids.add(grela_id)

            if i % 50 == 0:
                print(f"[{textsource}] processed {i} files from {dir_path}")
    conn.execute("COMMIT;")
except Exception:
    conn.execute("ROLLBACK;")
    raise

CPU times: user 8.42 s, sys: 2.34 s, total: 10.8 s
Wall time: 1.21 s


In [22]:
# set the work you want to remove
grela_id = "lagt_tlg1352.tlg001"   # e.g., "cc_12710"

conn.execute("BEGIN;")
try:
    conn.execute("DELETE FROM tokens    WHERE grela_id = ?", [grela_id])
    conn.execute("DELETE FROM sentences WHERE grela_id = ?", [grela_id])
    conn.execute("DELETE FROM works WHERE grela_id = ?", [grela_id])
    conn.execute("COMMIT;")
except Exception:
    conn.execute("ROLLBACK;")
    raise

In [23]:
conn.execute("UPDATE tokens SET token_id = rowid;")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [24]:
# 1) Mark untouched LAGT works as 'lagt3'
conn.execute("""
UPDATE works
SET textsource = 'lagt3'
WHERE grela_source = 'lagt'
  AND (textsource IS NULL OR textsource = '');
""")

# (optional) count before remap
to_remap = conn.execute("""
SELECT COUNT(*) AS cnt
FROM tokens t
JOIN works  w ON w.grela_id = t.grela_id
WHERE w.grela_source = 'lagt'
  AND w.textsource   = 'lagt3'
  AND length(t.pos)  = 1;
""").fetchone()[0]
print(f"Tokens to remap (single-letter) under lagt/lagt3: {to_remap:,}")

# 2) Remap POS for LAGT + lagt3 (single-letter only) — correlated subquery, no FROM
conn.execute("""
WITH m(pos1, ud) AS (
  VALUES
    ('n','NOUN'), ('v','VERB'), ('a','ADJ'), ('r','ADP'),
    ('p','PRON'), ('l','DET'),  ('c','CCONJ'), ('d','ADV'),
    ('u','PUNCT'),('g','PART'), ('z','X')
)
UPDATE tokens
SET pos = (SELECT ud FROM m WHERE m.pos1 = tokens.pos)
WHERE length(tokens.pos) = 1
  AND tokens.pos IN (SELECT pos1 FROM m)
  AND EXISTS (
        SELECT 1
        FROM works w
        WHERE w.grela_id     = tokens.grela_id
          AND w.grela_source = 'lagt'
          AND w.textsource   = 'lagt3'
  );
""")

# 3) After-summary
print(conn.execute("""
SELECT pos, COUNT(*) AS n
FROM tokens t
JOIN works w ON w.grela_id = t.grela_id
WHERE w.grela_source = 'lagt'
  AND w.textsource   = 'lagt3'
GROUP BY pos
ORDER BY n DESC;
""").fetchdf())

Tokens to remap (single-letter) under lagt/lagt3: 949
     pos     n
0   VERB  1210
1   NOUN  1169
2  PUNCT  1066
3      x   949
4  CCONJ   844
5    DET   824
6   PRON   670
7    ADJ   510
8    ADP   304


In [25]:
conn.execute("""--
UPDATE works AS w
SET    token_count = sub.cnt
FROM (
        SELECT grela_id, COUNT(*) AS cnt
        FROM tokens
        GROUP BY grela_id
     ) AS sub
WHERE w.grela_id = sub.grela_id;
""")

In [26]:
# Query to get table and column information
query = """
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    ORDER BY table_name, ordinal_position
"""

# Execute the query and fetch the schema information as a DataFrame
df = conn.execute(query).fetchdf()

# Group the schema details by table
tables = df.groupby("table_name")

# Markdown generation
markdown = "# Database Schema Documentation\n\n"
for table_name, group in tables:
    markdown += f"## Table: `{table_name}`\n\n"
    markdown += "| Column Name     | Data Type    | Is Nullable | Default Value |\n"
    markdown += "|-----------------|-------------|-------------|---------------|\n"

    for _, row in group.iterrows():
        markdown += (
            f"| {row['column_name']} | {row['data_type']} | "
            f"{row['is_nullable']} | {row['column_default'] or 'N/A'} |\n"
        )

    markdown += "\n"  # Add a space between tables


In [27]:
print(markdown)

# Database Schema Documentation

## Table: `sentence_embeddings`

| Column Name     | Data Type    | Is Nullable | Default Value |
|-----------------|-------------|-------------|---------------|
| sentence_id | VARCHAR | NO | N/A |
| grela_id | VARCHAR | YES | N/A |
| model | VARCHAR | YES | N/A |
| embedding | JSON | YES | N/A |

## Table: `sentences`

| Column Name     | Data Type    | Is Nullable | Default Value |
|-----------------|-------------|-------------|---------------|
| sentence_id | VARCHAR | YES | N/A |
| grela_id | VARCHAR | YES | N/A |
| position | INTEGER | YES | N/A |
| text | VARCHAR | YES | N/A |
| subwork_id | VARCHAR | YES | N/A |

## Table: `tokens`

| Column Name     | Data Type    | Is Nullable | Default Value |
|-----------------|-------------|-------------|---------------|
| sentence_id | VARCHAR | YES | N/A |
| grela_id | VARCHAR | YES | N/A |
| token_text | VARCHAR | YES | N/A |
| lemma | VARCHAR | YES | N/A |
| pos | VARCHAR | YES | N/A |
| char_start | IN

In [29]:
query = """
    SELECT t.*, w.*
    FROM tokens t
    JOIN works w ON t.grela_id = w.grela_id
    WHERE w.grela_id LIKE 'lagt_tlg2062.tlg050'
"""

gnt_tokens = conn.execute(query).fetchdf()
gnt_tokens.head(10)

,sentence_id,grela_id,token_text,lemma,pos,char_start,char_end,token_id,ref,grela_source,...,noscemus_discipline,title_short,emlap_noscemus_id,place_publication,place_geonames,author_viaf,title_viaf,date_random,token_count,textsource
0,lagt_tlg2062.tlg050_0,lagt_tlg2062.tlg050,Άπελθόντος,άπελθόντος,VERB,0,10,384872611,"{""div"": 0}",lagt,...,None,None,NaN,None,None,NaN,NaN,446.0,3611,exprecce
1,lagt_tlg2062.tlg050_0,lagt_tlg2062.tlg050,τοῦ,ὁ,DET,11,14,384872612,"{""div"": 0}",lagt,...,None,None,NaN,None,None,NaN,NaN,446.0,3611,exprecce
2,lagt_tlg2062.tlg050_0,lagt_tlg2062.tlg050,έπισκόπου,έπισκόπος,NOUN,15,24,384872613,"{""div"": 0}",lagt,...,None,None,NaN,None,None,NaN,NaN,446.0,3611,exprecce
3,lagt_tlg2062.tlg050_0,lagt_tlg2062.tlg050,μαρτύρων,μάρτυς,NOUN,25,33,384872614,"{""div"": 0}",lagt,...,None,None,NaN,None,None,NaN,NaN,446.0,3611,exprecce
4,lagt_tlg2062.tlg050_0,lagt_tlg2062.tlg050,ἡμέρα,ἡμέρα,NOUN,34,39,384872615,"{""div"": 0}",lagt,...,None,None,NaN,None,None,NaN,NaN,446.0,3611,exprecce
5,lagt_tlg2062.tlg050_0,lagt_tlg2062.tlg050,ἐν,ἐν,ADP,40,42,384872616,"{""div"": 0}",lagt,...,None,None,NaN,None,None,NaN,NaN,446.0,3611,exprecce
6,lagt_tlg2062.tlg050_0,lagt_tlg2062.tlg050,τῇ,ὁ,DET,43,45,384872617,"{""div"": 0}",lagt,...,None,None,NaN,None,None,NaN,NaN,446.0,3611,exprecce
7,lagt_tlg2062.tlg050_0,lagt_tlg2062.tlg050,χώρᾳ,χώρα,NOUN,46,50,384872618,"{""div"": 0}",lagt,...,None,None,NaN,None,None,NaN,NaN,446.0,3611,exprecce
8,lagt_tlg2062.tlg050_0,lagt_tlg2062.tlg050,ἐπιτελέσαι,ἐπιτελέω,VERB,51,61,384872619,"{""div"": 0}",lagt,...,None,None,NaN,None,None,NaN,NaN,446.0,3611,exprecce
9,lagt_tlg2062.tlg050_0,lagt_tlg2062.tlg050,",",",",PUNCT,61,62,384872620,"{""div"": 0}",lagt,...,None,None,NaN,None,None,NaN,NaN,446.0,3611,exprecce


In [30]:
conn.close()

In [7]:
#sents_emlap, tokens_emlap = process_sentences_from_dir(emlap_sents_data_dir, "emlap")